In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Comparing Airlines

In [ ]:
import pandas as pd
import numpy as np
import io

df = pd.read_csv('/content/drive/MyDrive/IDS567/Total_Data.csv')

In [ ]:
# Build route identifier
df["ROUTE"] = df["ORIGIN_AIRPORT_ID"].astype(str) + "-" + df["DEST_AIRPORT_ID"].astype(str)

# ── Per-airline metrics ───────────────────────────────────────────────────────
metrics = (
    df.groupby("OP_UNIQUE_CARRIER")
    .agg(
        total_flights=("ROUTE", "count"),
        avg_distance=("DISTANCE", "mean"),
        avg_dep_delay=("DEP_DELAY", "mean"),
        dep_del15_rate=("DEP_DEL15", "mean"),
        unique_origins=("ORIGIN_AIRPORT_ID", "nunique"),
    )
    .round(2)
)

# ── Shared routes with UA ─────────────────────────────────────────────────────
ua_routes = set(df.loc[df["OP_UNIQUE_CARRIER"] == "UA", "ROUTE"])

shared = {}
for carrier, group in df[df["OP_UNIQUE_CARRIER"] != "UA"].groupby("OP_UNIQUE_CARRIER"):
    shared[carrier] = len(ua_routes & set(group["ROUTE"]))

metrics["shared_routes_with_UA"] = pd.Series(shared)
metrics.loc["UA", "shared_routes_with_UA"] = len(ua_routes)

# ── UA Baseline ───────────────────────────────────────────────────────────────
ua = metrics.loc["UA"]
others = metrics.drop("UA").sort_values("total_flights", ascending=False)

print("=" * 75)
print("UNITED AIRLINES (UA) — BASELINE")
print("=" * 75)
print(f"  Total Flights       : {int(ua['total_flights']):,}")
print(f"  Avg Flight Distance : {ua['avg_distance']:,.2f} mi")
print(f"  Avg Dep Delay       : {ua['avg_dep_delay']:.2f} min")
print(f"  DEP_DEL15 Rate      : {ua['dep_del15_rate']*100:.2f}%")
print(f"  Unique Origins      : {int(ua['unique_origins'])}")
print(f"  Total Unique Routes : {int(ua['shared_routes_with_UA'])}")

# ── Comparison Table ──────────────────────────────────────────────────────────
print("\n" + "=" * 75)
print("COMPARISON: OTHER AIRLINES vs UA")
print("=" * 75)
print(f"{'Carrier':<10}{'Flights':>10}{'AvgDist':>12}{'AvgDelay':>12}{'Del15%':>10}{'Origins':>10}{'SharedRoutes':>14}")
print("-" * 75)
for carrier, row in metrics.sort_values("total_flights", ascending=False).iterrows():
    diff_delay = row["avg_dep_delay"] - ua["avg_dep_delay"]
    diff_del15 = (row["dep_del15_rate"] - ua["dep_del15_rate"]) * 100
    shared_count = int(row["shared_routes_with_UA"]) if not np.isnan(row["shared_routes_with_UA"]) else 0
    print(
        f"{carrier:<10}"
        f"{int(row['total_flights']):>10,}"
        f"{row['avg_distance']:>12.2f}"
        f"{row['avg_dep_delay']:>10.2f} ({diff_delay:+.2f})"
        f"{row['dep_del15_rate']*100:>8.2f}% ({diff_del15:+.2f}pp)"
        f"{int(row['unique_origins']):>10}"
        f"{shared_count:>14,}"
    )

print("-" * 75)
print("Note: delay diff = vs UA (positive = worse, negative = better)")

# ── Clean summary DataFrame ───────────────────────────────────────────────────
print("\nFull metrics table:")
display(metrics.sort_values("total_flights", ascending=False))


UNITED AIRLINES (UA) — BASELINE
  Total Flights       : 62,007
  Avg Flight Distance : 1,119.43 mi
  Avg Dep Delay       : 8.40 min
  DEP_DEL15 Rate      : 17.00%
  Unique Origins      : 119
  Total Unique Routes : 811

COMPARISON: OTHER AIRLINES vs UA
Carrier      Flights     AvgDist    AvgDelay    Del15%   Origins  SharedRoutes
---------------------------------------------------------------------------
WN           105,307      751.57      7.30 (-1.10)   17.00% (+0.00pp)       104           173
DL            76,306      978.75     12.97 (+4.57)   20.00% (+3.00pp)       140            89
AA            75,088      992.63     13.00 (+4.60)   19.00% (+2.00pp)       119           155
OO            65,036      462.38     13.07 (+4.67)   19.00% (+2.00pp)       239           218
UA            62,007     1119.43      8.40 (+0.00)   17.00% (+0.00pp)       119           811
YX            27,833      507.04      2.64 (-5.76)   12.00% (-5.00pp)        83           139
MQ            21,890      53

,total_flights,avg_distance,avg_dep_delay,dep_del15_rate,unique_origins,shared_routes_with_UA
OP_UNIQUE_CARRIER,,,,,,
WN,105307,751.57,7.30,0.17,104,173.0
DL,76306,978.75,12.97,0.20,140,89.0
AA,75088,992.63,13.00,0.19,119,155.0
OO,65036,462.38,13.07,0.19,239,218.0
UA,62007,1119.43,8.40,0.17,119,811.0
YX,27833,507.04,2.64,0.12,83,139.0
MQ,21890,539.12,9.01,0.16,145,92.0
OH,21094,440.71,17.68,0.23,94,8.0
AS,18163,1349.81,5.26,0.17,85,70.0


In [ ]:
compare = metrics.loc[["UA", "AA"]]
display(compare)

,total_flights,avg_distance,avg_dep_delay,dep_del15_rate,unique_origins,shared_routes_with_UA
OP_UNIQUE_CARRIER,,,,,,
UA,62007,1119.43,8.4,0.17,119,811.0
AA,75088,992.63,13.0,0.19,119,155.0


We should compare UA to AA.

1) They both have 119 airports as unique orgins

2) similar flight  profiles (UA 1193 vs AA 993)

3) similar flight volume (AA 75K vs UA 62K)